# Refine ESCO Skills

Load ESCO occupations with NACE codes and merge with skills from the ESCO skills relation table. This step prepares the data for refining and selecting the most relevant skills for each occupation.

**Note:** This notebook explores the skills refinement process. Future implementation will move reusable logic to `src/`.

This notebook contains the exploration and can be used for testing individual projects.

## 0. Setup

### 0.01 Import Required Libraries

In [1]:
import pandas as pd
from pathlib import Path

# Import our config
import sys
sys.path.append(str(Path.cwd().parent))
from src.config import load_config

### 0.02 Load Configuration

In [2]:
# Get project root
project_root = Path.cwd().parent

# Load project config
config = load_config()

print("✓ Configuration loaded")

✓ Configuration loaded


### 0.03 Set Up Paths

In [3]:
# Project ID to use for this exploration
PROJECT_ID = "P511453"

# Set up paths
unique_esco_nace_dir = project_root / "data" / "silver" / "unique_esco_nace_csv"
esco_skills_file = project_root / "data" / "bronze" / "esco" / "occupationSkillRelations_en.csv"

# Input file: unique ESCO occupations with NACE codes
unique_esco_nace_file = unique_esco_nace_dir / f"{PROJECT_ID}_unique_matched_with_nace.csv"

print(f"Project ID: {PROJECT_ID}")
print(f"Input file: {unique_esco_nace_file}")
print(f"ESCO skills file: {esco_skills_file}")

Project ID: P511453
Input file: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv/P511453_unique_matched_with_nace.csv
ESCO skills file: /Users/lauren/repos/PAD2Skills/data/bronze/esco/occupationSkillRelations_en.csv


## 1. Load and Merge Data

### 1.01 Load Unique ESCO Occupations with NACE Codes

In [4]:
# Load the unique ESCO occupations with NACE codes
df_occupations = pd.read_csv(unique_esco_nace_file)

print(f"Loaded {len(df_occupations)} unique ESCO occupations")
print(f"Columns: {list(df_occupations.columns)}")
print(f"\nFirst few rows:")
df_occupations.head()

Loaded 44 unique ESCO occupations
Columns: ['esco_id', 'esco_label', 'esco_description', 'group_code', 'group_label_en', 'division_code', 'division_label_en', 'section_code', 'section_label_en', 'pad_occupations', 'pad_activities', 'pad_skills', 'pad_quotes']

First few rows:


,esco_id,esco_label,esco_description,group_code,group_label_en,division_code,division_label_en,section_code,section_label_en,pad_occupations,pad_activities,pad_skills,pad_quotes
0,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ..."
1,0752ed49-03e7-4d75-8314-a051b3771a1d,public administration manager,"Public administration managers direct, monitor...",841,84.1 Administration of the State and the econo...,84,84 Public administration and defence; compulso...,P,P PUBLIC ADMINISTRATION AND DEFENCE; COMPULSOR...,"""head of department""","""Be competitively recruited and take part in o...","""performance management"", ""operational leaders...","V. KEY RISKS: ""Operationalization initiated: ..."
2,0ba06640-e0ac-4911-9e43-289a8e41651e,corporate trainer,"Corporate trainers train, coach, and guide emp...",855,85.5 Other education,85,85 Education,Q,Q EDUCATION,"""capacity building specialist""","""Design and implement capacity-building progra...","""training program design"", ""adult learning tec...","V. KEY RISKS: ""Eligible expenditures include: ..."
3,13d1b2b4-99dd-44da-9734-c9f74bae18f7,customer service representative,Customer service representatives handle compla...,822,82.2 Activities of call centres,82,"82 Office administrative, office support and o...",O,O ADMINISTRATIVE AND SUPPORT SERVICE ACTIVITIES,"""grievance officer""","""Operate the National Grievance Redress Mechan...","""grievance intake"", ""case management"", ""record...","IV. PROJECT APPRAISAL SUMMARY: ""The project wi..."
4,16b974f4-cb8c-436c-8c05-a16957c40131,fossil-fuel power plant operator,Fossil-fuel power plant operators operate and ...,351,"35.1 Electric power generation, transmission a...",35,"35 Electricity, gas, steam and air conditionin...",D,"D ELECTRICITY, GAS, STEAM AND AIR CONDITIONING...","""diesel generator operator""","""Operate and manage diesel gensets that will m...","""diesel genset operation"", ""routine inspection...","ANNEX 3: ECONOMIC AND FINANCIAL ANALYSIS: ""The..."


### 1.02 Load ESCO Skills Relations

In [5]:
# Load the ESCO occupation-skill relations
df_skills = pd.read_csv(esco_skills_file)

print(f"Loaded {len(df_skills)} occupation-skill relations")
print(f"Columns: {list(df_skills.columns)}")
print(f"\nFirst few rows:")
df_skills.head()

Loaded 126051 occupation-skill relations
Columns: ['occupationUri', 'occupationLabel', 'relationType', 'skillType', 'skillUri', 'skillLabel']

First few rows:


,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,knowledge,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques
1,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals
2,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/271a36a0-bc7a...,write risk assessment on performing arts produ...
3,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/47ed1d37-971b...,coordinate with creative departments
4,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/591dd514-735b...,adapt to artists' creative demands


### 1.03 Merge Occupations with Skills

In [6]:
# Merge occupations with skills (left one-to-many)
# The URIs are in different formats, so we need to extract the UUID from the skills file
# Skills file has: http://data.europa.eu/esco/occupation/{uuid}
# Occupations file has: {uuid}

# Extract UUID from skills file occupationUri
df_skills['occupation_uuid'] = df_skills['occupationUri'].str.split('/').str[-1]

# Now merge on the UUID
df_merged = df_occupations.merge(
    df_skills,
    left_on='esco_id',
    right_on='occupation_uuid',
    how='left'
)

print(f"Merged data shape: {df_merged.shape}")
print(f"Original occupations: {len(df_occupations)}")
print(f"After merge: {len(df_merged)} rows (includes multiple skills per occupation)")
print(f"\nColumns in merged data:")
print(list(df_merged.columns))
print(f"\nSample of merged data:")
df_merged.head(10)

Merged data shape: (2210, 20)
Original occupations: 44
After merge: 2210 rows (includes multiple skills per occupation)

Columns in merged data:
['esco_id', 'esco_label', 'esco_description', 'group_code', 'group_label_en', 'division_code', 'division_label_en', 'section_code', 'section_label_en', 'pad_occupations', 'pad_activities', 'pad_skills', 'pad_quotes', 'occupationUri', 'occupationLabel', 'relationType', 'skillType', 'skillUri', 'skillLabel', 'occupation_uuid']

Sample of merged data:


,esco_id,esco_label,esco_description,group_code,group_label_en,division_code,division_label_en,section_code,section_label_en,pad_occupations,pad_activities,pad_skills,pad_quotes,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel,occupation_uuid
0,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ...",http://data.europa.eu/esco/occupation/05ebeb56...,bridge construction supervisor,essential,knowledge,http://data.europa.eu/esco/skill/44297bbb-306a...,mechanical tools,05ebeb56-6ceb-488d-ba91-ed15088efc8e
1,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ...",http://data.europa.eu/esco/occupation/05ebeb56...,bridge construction supervisor,essential,skill/competence,http://data.europa.eu/esco/skill/02e0b3e9-c9e0...,keep records of bridge investigation findings,05ebeb56-6ceb-488d-ba91-ed15088efc8e
2,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ...",http://data.europa.eu/esco/occupation/05ebeb56...,bridge construction supervisor,essential,skill/competence,http://data.europa.eu/esco/skill/3892f2ff-714e...,check compatibility of materials,05ebeb56-6ceb-488d-ba91-ed15088efc8e
3,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ...",http://data.europa.eu/esco/occupation/05ebeb56...,bridge construction supervisor,essential,skill/competence,http://data.europa.eu/esco/skill/41a3240f-58ba...,conduct quality control analysis,05ebeb56-6ceb-488d-ba91-ed15088efc8e
4,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ...",http://data.europa.eu/esco/occupation/05ebeb56...,bridge construction supervisor,essential,skill/competence,http://data.europa.eu/esco/skill/44e2c4c4-9fd2...,manage health and safety standards,05ebeb56-6ceb-488d-ba91-ed15088efc8e
5,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""","""Supervise construction and installation of fl...","""construction management"", ""site inspection"", ...","ANNEX 4: CLIMATE RISKS 46: ""To enhance the ...",http://data.europa.eu/esco/occupation/05ebeb56...,bridge construction supervisor,essential,skill/competence,http://data.europa.eu/esco/skill/44f6a65f-6be7...,monitor stock level,05ebeb56-6ceb-488d-ba91-ed15088efc8e
6,05ebeb56-6ceb-488d-b

### 1.04 Summary Statistics

In [7]:
# Filter out skills that are not essential
df_merged = df_merged[df_merged['relationType'] == 'essential']

# Count skills per occupation
skills_per_occupation = df_merged.groupby('esco_id').size().reset_index(name='skill_count')

print("Skills per occupation statistics:")
print(skills_per_occupation['skill_count'].describe())

print(f"\n\nTop 5 occupations with most skills:")
top_occupations = skills_per_occupation.nlargest(5, 'skill_count')
for _, row in top_occupations.iterrows():
    esco_id = row['esco_id']
    count = row['skill_count']
    label = df_occupations[df_occupations['esco_id'] == esco_id]['esco_label'].values[0]
    print(f"  {label}: {count} skills")

print(f"\n\nRelation types:")
print(df_merged['relationType'].value_counts())

print(f"\n\nSkill types:")
print(df_merged['skillType'].value_counts())

Skills per occupation statistics:
count    44.000000
mean     21.886364
std      10.675370
min       8.000000
25%      15.750000
50%      20.000000
75%      26.500000
max      72.000000
Name: skill_count, dtype: float64


Top 5 occupations with most skills:
  consultant social worker: 72 skills
  project manager: 36 skills
  data analyst: 36 skills
  equality and inclusion manager: 35 skills
  monitoring and evaluation officer: 33 skills


Relation types:
relationType
essential    963
Name: count, dtype: int64


Skill types:
skillType
skill/competence    690
knowledge           268
Name: count, dtype: int64


In [8]:
# Look at skills for the first occupation
sample_occupation = df_occupations.iloc[0]
sample_esco_id = sample_occupation['esco_id']
sample_label = sample_occupation['esco_label']

print(f"Occupation: {sample_label}")
print(f"ESCO ID: {sample_esco_id}")
print(f"Description: {sample_occupation['esco_description'][:200]}...")
print(f"\nNACE Codes: {sample_occupation['group_label_en']}")

# Get skills for this occupation
occupation_skills = df_merged[df_merged['esco_id'] == sample_esco_id][
    ['skillLabel', 'relationType', 'skillType']
].sort_values(['relationType', 'skillType'])

print(f"\n\nSkills ({len(occupation_skills)} total):")
print("\nEssential skills:")
essential = occupation_skills[occupation_skills['relationType'] == 'essential']
for idx, row in essential.head(10).iterrows():
    print(f"  [{row['skillType']}] {row['skillLabel']}")

print(f"\nOptional skills:")
optional = occupation_skills[occupation_skills['relationType'] == 'optional']
for idx, row in optional.head(10).iterrows():
    print(f"  [{row['skillType']}] {row['skillLabel']}")

Occupation: bridge construction supervisor
ESCO ID: 05ebeb56-6ceb-488d-ba91-ed15088efc8e
Description: Bridge construction supervisors monitor the construction of bridges. They assign tasks and take quick decisions to resolve problems....

NACE Codes: 42.1 Construction of roads and railways


Skills (28 total):

Essential skills:
  [knowledge] mechanical tools
  [skill/competence] keep records of bridge investigation findings
  [skill/competence] check compatibility of materials
  [skill/competence] conduct quality control analysis
  [skill/competence] manage health and safety standards
  [skill/competence] monitor stock level
  [skill/competence] work in a construction team
  [skill/competence] inspect construction supplies
  [skill/competence] manage bridge construction projects
  [skill/competence] ensure compliance with construction project deadline

Optional skills:


## 2. Construct JSON Input

### 2.01 Import Project Summary

In [9]:
# Load the project summary
pad_summaries_dir = project_root / "data" / "silver" / "pad_summaries"
summary_file = pad_summaries_dir / f"{PROJECT_ID}_summary.txt"

# Read the summary
with open(summary_file, 'r') as f:
    project_summary = f.read()

print(f"Loaded project summary from: {summary_file}")
print(f"Summary length: {len(project_summary)} characters")
print(f"\nFirst 500 characters:")
print(project_summary[:500])

Loaded project summary from: /Users/lauren/repos/PAD2Skills/data/silver/pad_summaries/P511453_summary.txt
Summary length: 1932 characters

First 500 characters:
The Guinea Electricity Access Scale Up Project-Phase 2 (GNEAP-2) will expand electricity access in selected urban and rural areas of Guinea. It is a US$271.8 million investment project financing with performance-based conditions. Financing includes US$132.3 million from IDA, US$100.0 million in parallel co-financing from the European Investment Bank (EIB), and US$11.7 million of private capital mobilized. Guinea faces rapid demand growth, heavy reliance on climate-sensitive hydropower, and weak 


### 2.02 Extract Skill Code from URI

In [10]:
# Extract skill_code from skillUri (the part after the last slash)
df_merged['skill_code'] = df_merged['skillUri'].str.split('/').str[-1]

print(f"Added skill_code column")
print(f"\nSample skill codes:")
print(df_merged[['skillUri', 'skill_code']].head())
print(f"\nUnique skill codes: {df_merged['skill_code'].nunique()}")

Added skill_code column

Sample skill codes:
                                            skillUri  \
0  http://data.europa.eu/esco/skill/44297bbb-306a...   
1  http://data.europa.eu/esco/skill/02e0b3e9-c9e0...   
2  http://data.europa.eu/esco/skill/3892f2ff-714e...   
3  http://data.europa.eu/esco/skill/41a3240f-58ba...   
4  http://data.europa.eu/esco/skill/44e2c4c4-9fd2...   

                             skill_code  
0  44297bbb-306a-46fa-ab1b-fa3fb327ebda  
1  02e0b3e9-c9e0-4063-983b-b5a7697e0f78  
2  3892f2ff-714e-4201-823d-5d6fc5bab621  
3  41a3240f-58ba-44e2-a688-edeb21ea5c16  
4  44e2c4c4-9fd2-42d4-a133-d81360ded4bc  

Unique skill codes: 711


### 2.03 Create ESCO Occupation Number

In [11]:
# Sort by esco_id and skill_code
df_merged_sorted = df_merged.sort_values(['esco_id', 'skill_code']).reset_index(drop=True)

# Create a three-digit ordered ID for each unique esco_id
unique_esco_ids = df_merged_sorted['esco_id'].unique()
esco_num_map = {esco_id: f"{i+1:03d}" for i, esco_id in enumerate(unique_esco_ids)}

# Add the esco_num column
df_merged_sorted['esco_num'] = df_merged_sorted['esco_id'].map(esco_num_map)

print(f"Created esco_num for {len(unique_esco_ids)} unique occupations")
print(f"\nSample mapping:")
for i, (esco_id, esco_num) in enumerate(list(esco_num_map.items())[:5]):
    label = df_occupations[df_occupations['esco_id'] == esco_id]['esco_label'].values[0]
    print(f"  {esco_num}: {label}")

print(f"\nMerged data with esco_num:")
print(df_merged_sorted[['esco_num', 'esco_id', 'esco_label', 'skill_code', 'skillLabel']].head(20))

Created esco_num for 44 unique occupations

Sample mapping:
  001: bridge construction supervisor
  002: public administration manager
  003: corporate trainer
  004: customer service representative
  005: fossil-fuel power plant operator

Merged data with esco_num:
   esco_num                               esco_id  \
0       001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
1       001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
2       001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
3       001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
4       001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
5       001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
6       001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
7       001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
8       001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
9       001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
10      001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
11      001  05ebeb56-6ceb-488d-ba91-ed15088efc8e   
12      001  05ebeb56-6ceb-488d-ba91-ed15088

### 2.04 Inspect Available Columns

Before chunking, let's verify what columns we have for PAD data (occupations, activities, skills).

In [12]:
# Check what columns we have in the original occupations data
print("Columns in df_occupations:")
print(list(df_occupations.columns))

print("\n\nColumns in df_merged_sorted:")
print(list(df_merged_sorted.columns))

# Look for columns that might contain PAD data
print("\n\nSample row from df_occupations:")
print(df_occupations.iloc[0])

Columns in df_occupations:
['esco_id', 'esco_label', 'esco_description', 'group_code', 'group_label_en', 'division_code', 'division_label_en', 'section_code', 'section_label_en', 'pad_occupations', 'pad_activities', 'pad_skills', 'pad_quotes']


Columns in df_merged_sorted:
['esco_id', 'esco_label', 'esco_description', 'group_code', 'group_label_en', 'division_code', 'division_label_en', 'section_code', 'section_label_en', 'pad_occupations', 'pad_activities', 'pad_skills', 'pad_quotes', 'occupationUri', 'occupationLabel', 'relationType', 'skillType', 'skillUri', 'skillLabel', 'occupation_uuid', 'skill_code', 'esco_num']


Sample row from df_occupations:
esco_id                           05ebeb56-6ceb-488d-ba91-ed15088efc8e
esco_label                              bridge construction supervisor
esco_description     Bridge construction supervisors monitor the co...
group_code                                                         421
group_label_en                 42.1 Construction of ro

### 2.05 Create JSON Chunks (3 Occupations per Chunk)

We'll chunk the data by grouping 3 unique ESCO occupations together. Each chunk will contain the project info and an array of occupations with their skills.

In [13]:
import json

# Get unique ESCO IDs in order
unique_esco_ids = df_merged_sorted['esco_id'].unique()

# Define chunk size
CHUNK_SIZE = 3

# Split into chunks of 3 occupations
def chunk_list(lst, chunk_size):
    """Split a list into chunks of specified size."""
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

# Create chunks
esco_chunks = list(chunk_list(unique_esco_ids, CHUNK_SIZE))

print(f"Total unique occupations: {len(unique_esco_ids)}")
print(f"Number of chunks (3 occupations each): {len(esco_chunks)}")
print(f"Last chunk size: {len(esco_chunks[-1])}")
print(f"\nFirst chunk ESCO IDs:")
for esco_id in esco_chunks[0]:
    label = df_occupations[df_occupations['esco_id'] == esco_id]['esco_label'].values[0]
    print(f"  - {label}")

Total unique occupations: 44
Number of chunks (3 occupations each): 15
Last chunk size: 2

First chunk ESCO IDs:
  - bridge construction supervisor
  - public administration manager
  - corporate trainer


### 2.06 Build JSON Structure for Each Chunk

Create the JSON structure with project info and occupations array. Each occupation includes its ESCO skills.

In [14]:
def create_json_chunk(chunk_esco_ids, df_data, df_occ, project_id, project_summary):
    """
    Create a JSON structure for a chunk of occupations.
    
    Args:
        chunk_esco_ids: List of ESCO IDs in this chunk
        df_data: DataFrame with merged occupation and skill data (df_merged_sorted)
        df_occ: DataFrame with occupation details (df_occupations)
        project_id: Project identifier
        project_summary: Project summary text
    
    Returns:
        Dictionary representing the JSON structure
    """
    occupations = []
    
    for esco_id in chunk_esco_ids:
        # Get occupation details
        occ_row = df_occ[df_occ['esco_id'] == esco_id].iloc[0]
        
        # Get all skills for this occupation
        occ_skills = df_data[df_data['esco_id'] == esco_id]
        
        # Build skills list (each skill as a separate dict)
        skills_list = []
        for _, skill_row in occ_skills.iterrows():
            skills_list.append({
                'skill_code': skill_row['skill_code'],
                'skill_label': skill_row['skillLabel']
            })
        
        # Build occupation dict
        # Note: Update these column names based on what's actually in your data
        occupation_dict = {
            'esco_num': occ_skills.iloc[0]['esco_num'],  # Get esco_num from the merged data
            'esco_id': esco_id,
            'esco_label': occ_row['esco_label'],
            'pad_occupations': occ_row.get('pad_occupations', ''),  # Update column name as needed
            'pad_activities': occ_row.get('pad_activities', ''),  # Update column name as needed
            'pad_skills': occ_row.get('pad_skills', ''),  # Update column name as needed
            'skills': skills_list
        }
        
        occupations.append(occupation_dict)
    
    # Build the final JSON structure
    json_chunk = {
        'project_id': project_id,
        'project_summary': project_summary,
        'occupations': occupations
    }
    
    return json_chunk

In [15]:
# Test with the first chunk (commented out)
# if 0:
#     first_chunk_json = create_json_chunk(
#         esco_chunks[0], 
#         df_merged_sorted, 
#         df_occupations,
#         PROJECT_ID,
#         project_summary
#     )
#
#     print("Sample JSON structure:")
#     print(json.dumps(first_chunk_json, indent=2, ensure_ascii=False))
#     print("\n...")
#     print(f"\nTotal occupations in chunk: {len(first_chunk_json['occupations'])}")
#     print(f"Skills in first occupation: {len(first_chunk_json['occupations'][0]['skills'])}")

## 3. Call API to Evaluate Skills

### 3.01 Load Environment and Initialize OpenAI Client

In [16]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
env_path = project_root / ".env"

if not env_path.exists():
    raise FileNotFoundError(
        f"'.env' file not found at {env_path}\n"
        "Please copy .env.example to .env and add your OpenAI API key."
    )

# Load from specific path
load_dotenv(env_path, override=True)

# Get OpenAI API key from environment
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Verify API key is set
if not OPENAI_API_KEY:
    raise ValueError("Missing required environment variable: OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

print("✓ Environment variables loaded")
print(f"  API Key: {OPENAI_API_KEY[:10]}...{OPENAI_API_KEY[-4:]}")
print("✓ OpenAI client initialized")

✓ Environment variables loaded
  API Key: sk-proj-cj...__0A
✓ OpenAI client initialized


### 3.02 Create Function to Call API for Skills Evaluation

In [17]:
def evaluate_skills_with_api(chunk_json, client):
    """
    Call OpenAI API to evaluate skills for a chunk of occupations.
    
    Args:
        chunk_json: Dictionary containing the input JSON structure
        client: OpenAI client instance
    
    Returns:
        Dictionary with evaluation results
    """
    # Convert to JSON string for input message
    input_message = json.dumps(chunk_json, indent=2, ensure_ascii=False)
    
    print(f"Calling API...")
    print(f"  Input size: {len(input_message):,} chars")
    print(f"  Occupations: {len(chunk_json['occupations'])}")
    
    # Call OpenAI API with prompt
    response = client.responses.create(
        prompt={
            "id": "pmpt_695c42b522b8819683a2e305c9886b2a0a31f94fbd8aa0d8",
            "version": "1"
        },
        input=[
            {"role": "user", "content": input_message}
        ],
        reasoning={
            "summary": None
        },
        store=False,
        include=[
            "reasoning.encrypted_content",
            "web_search_call.action.sources"
        ]
    )
    
    # Extract the text from the response
    result_text = None
    for item in response.output:
        if hasattr(item, 'content') and hasattr(item, 'role'):
            result_text = item.content[0].text
            break
    
    if result_text is None:
        raise ValueError("No content found in API response")
    
    # Parse the JSON response
    result_json = json.loads(result_text)
    
    print(f"  ✓ Response received (response_id: {response.id})")
    print(f"  Output occupations: {len(result_json['occupations'])}")
    
    return result_json

print("✓ Function defined: evaluate_skills_with_api()")

✓ Function defined: evaluate_skills_with_api()


### 3.03 Test with First Chunk

In [18]:
# Test with first chunk (commented out)
# if 0:
#     result_json = evaluate_skills_with_api(first_chunk_json, client)
#
#     print("\n" + "=" * 80)
#     print("API Result Sample:")
#     print("=" * 80)
#     print(json.dumps(result_json, indent=2)[:1500])
#     print("\n...")
#     print(f"\nProject ID: {result_json['project_id']}")
#     print(f"Occupations returned: {len(result_json['occupations'])}")
#     print(f"First occupation skills: {len(result_json['occupations'][0]['skills'])}")

### 3.04 Convert Result to DataFrame

In [19]:
# Convert the nested JSON result to a flat dataframe
def result_to_dataframe(result_json):
    """
    Convert API result JSON to a flat DataFrame.
    
    Args:
        result_json: Dictionary with project_id and occupations array
    
    Returns:
        DataFrame with columns: esco_id, skill_code, relevant, top_five
    """
    rows = []
    
    for occupation in result_json['occupations']:
        esco_id = occupation['esco_id']
        
        for skill in occupation['skills']:
            rows.append({
                'esco_id': esco_id,
                'skill_code': skill['skill_code'],
                'relevant': skill['relevant'],
                'top_five': skill['top_five']
            })
    
    return pd.DataFrame(rows)

# Convert result to dataframe (commented out)
# if 0:
#     df_evaluation = result_to_dataframe(result_json)
#
#     print(f"Evaluation DataFrame shape: {df_evaluation.shape}")
#     print(f"\nColumns: {list(df_evaluation.columns)}")
#     print(f"\nSample:")
#     df_evaluation.head(20)

In [20]:
# Check evaluation results per occupation (commented out)
# if 0:
#     print("Evaluation Statistics:")
#     print("=" * 80)
#
#     # Get summary statistics
#     total_occupations = df_with_evaluation['esco_id'].nunique()
#     total_skills = len(df_with_evaluation)
#     relevant_skills = df_with_evaluation['relevant'].sum()
#     top_five_skills = df_with_evaluation['top_five'].sum()
#
#     print(f"\nOverall Statistics:")
#     print(f"  Total occupations: {total_occupations}")
#     print(f"  Total skills: {total_skills}")
#     print(f"  Relevant skills: {relevant_skills} ({relevant_skills/total_skills*100:.1f}%)")
#     print(f"  Top five skills: {top_five_skills}")
#     print(f"  Average skills per occupation: {total_skills/total_occupations:.1f}")
#     print(f"  Average relevant skills per occupation: {relevant_skills/total_occupations:.1f}")
#
#     # Show sample occupations
#     print("\n" + "=" * 80)
#     print("Sample Occupation Details (first 5 occupations):")
#     print("=" * 80)
#
#     sample_esco_ids = df_with_evaluation['esco_id'].unique()[:5]
#
#     for esco_id in sample_esco_ids:
#         occ_data = df_with_evaluation[df_with_evaluation['esco_id'] == esco_id]
#         label = occ_data.iloc[0]['esco_label']
#         
#         total_skills = len(occ_data)
#         relevant_count = occ_data['relevant'].sum()
#         top_five_count = occ_data['top_five'].sum()
#         
#         print(f"\n{label}")
#         print(f"  Total skills: {total_skills}")
#         print(f"  Relevant: {relevant_count} ({relevant_count/total_skills*100:.1f}%)")
#         print(f"  Top five: {top_five_count}")
#         
#         # Show top five skills
#         top_five_skills = occ_data[occ_data['top_five'] == True]['skillLabel'].tolist()
#         if top_five_skills:
#             print(f"  Top 5 skills:")
#             for i, skill in enumerate(top_five_skills, 1):
#                 print(f"    {i}. {skill}")

## 4. Process all chunks

### 4.1 Call API and store result

In [ ]:
# Process all chunks and collect results
all_evaluation_results = []

# For testing, process only first 3 chunks
#chunks_to_process = esco_chunks[0:3]
chunks_to_process = esco_chunks

print(f"Processing {len(chunks_to_process)} chunks (out of {len(esco_chunks)} total)...")
print("=" * 80)

for i, chunk_esco_ids in enumerate(chunks_to_process, 1):
    print(f"\nChunk {i}/{len(chunks_to_process)}: {len(chunk_esco_ids)} occupations")
    
    # Create JSON for this chunk
    chunk_json = create_json_chunk(
        chunk_esco_ids,
        df_merged_sorted,
        df_occupations,
        PROJECT_ID,
        project_summary
    )
    
    # Call API to evaluate skills for this chunk
    result_json = evaluate_skills_with_api(chunk_json, client)
    
    # Convert result to dataframe
    df_chunk_evaluation = result_to_dataframe(result_json)
    
    print(f"  Evaluation rows: {len(df_chunk_evaluation)}")
    
    # Store the evaluation results
    all_evaluation_results.append(df_chunk_evaluation)

# Combine all evaluation results into a single dataframe
df_all_evaluations = pd.concat(all_evaluation_results, ignore_index=True)

print("\n" + "=" * 80)
print(f"Total evaluation rows: {len(df_all_evaluations)}")
print(f"Unique occupations evaluated: {df_all_evaluations['esco_id'].nunique()}")
print(f"Columns: {list(df_all_evaluations.columns)}")


Processing 3 chunks (out of 15 total)...

Chunk 1/3: 3 occupations
Calling API...
  Input size: 12,181 chars
  Occupations: 3
  ✓ Response received (response_id: resp_050d4d84e0c063eb01695c588edac88194b5a18b4112ac876f)
  Output occupations: 3
  Evaluation rows: 61

Chunk 2/3: 3 occupations
Calling API...
  Input size: 19,825 chars
  Occupations: 3
  ✓ Response received (response_id: resp_0b60575b4a66a51401695c58ce4ddc819697ffbcc1fb38a6e2)
  Output occupations: 3
  Evaluation rows: 112

Chunk 3/3: 3 occupations
Calling API...
  Input size: 12,297 chars
  Occupations: 3
  ✓ Response received (response_id: resp_0d71ba4d28f078b901695c597dd4448197a44c47bfb62dd461)
  Output occupations: 3
  Evaluation rows: 53

Total evaluation rows: 226
Unique occupations evaluated: 9
Columns: ['esco_id', 'skill_code', 'relevant', 'top_five']


### 4.02 Merge back to dataframe

In [23]:

# Merge the evaluation results back to the main dataframe
df_with_evaluation = df_merged_sorted.merge(
    df_all_evaluations,
    on=['esco_id', 'skill_code'],
    how='left'
)

print(f"\nOriginal df_merged_sorted shape: {df_merged_sorted.shape}")
print(f"All evaluations df shape: {df_all_evaluations.shape}")
print(f"Merged df_with_evaluation shape: {df_with_evaluation.shape}")

print(f"\nNew columns added: {[col for col in df_with_evaluation.columns if col not in df_merged_sorted.columns]}")

print(f"\nSample with evaluation columns:")
df_with_evaluation[['esco_num', 'esco_label', 'skill_code', 'skillLabel', 'relevant', 'top_five']].head(20)


Original df_merged_sorted shape: (963, 22)
All evaluations df shape: (226, 4)
Merged df_with_evaluation shape: (963, 24)

New columns added: ['relevant', 'top_five']

Sample with evaluation columns:


,esco_num,esco_label,skill_code,skillLabel,relevant,top_five
0,001,bridge construction supervisor,02e0b3e9-c9e0-4063-983b-b5a7697e0f78,keep records of bridge investigation findings,False,False
1,001,bridge construction supervisor,3892f2ff-714e-4201-823d-5d6fc5bab621,check compatibility of materials,True,False
2,001,bridge construction supervisor,41a3240f-58ba-44e2-a688-edeb21ea5c16,conduct quality control analysis,True,True
3,001,bridge construction supervisor,44297bbb-306a-46fa-ab1b-fa3fb327ebda,mechanical tools,True,False
4,001,bridge construction supervisor,44e2c4c4-9fd2-42d4-a133-d81360ded4bc,manage health and safety standards,True,True
5,001,bridge construction supervisor,44f6a65f-6be7-4818-9a66-73908797283f,monitor stock level,True,False
6,001,bridge construction supervisor,4fa2d6ba-4fff-4f21-bddd-4a54ae840955,work in a construction team,True,False
7,001,bridge construction supervisor,50f2d23c-b3fb-48a5-967b-14fd0a6dd205,inspect construction supplies,True,True
8,001,bridge construction supervisor,6101480a-811d-46da-b4f6-fc0360826f22,manage bridge construction projects,True,False
9,001,bridge construction supervisor,6181f475-110c-497c-a46d-ed2e14e7bc90,ensure compliance with construction project de...,True,False


In [24]:
df_with_evaluation.head(100)

,esco_id,esco_label,esco_description,group_code,group_label_en,division_code,division_label_en,section_code,section_label_en,pad_occupations,...,occupationLabel,relationType,skillType,skillUri,skillLabel,occupation_uuid,skill_code,esco_num,relevant,top_five
0,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""",...,bridge construction supervisor,essential,skill/competence,http://data.europa.eu/esco/skill/02e0b3e9-c9e0...,keep records of bridge investigation findings,05ebeb56-6ceb-488d-ba91-ed15088efc8e,02e0b3e9-c9e0-4063-983b-b5a7697e0f78,001,False,False
1,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""",...,bridge construction supervisor,essential,skill/competence,http://data.europa.eu/esco/skill/3892f2ff-714e...,check compatibility of materials,05ebeb56-6ceb-488d-ba91-ed15088efc8e,3892f2ff-714e-4201-823d-5d6fc5bab621,001,True,False
2,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""",...,bridge construction supervisor,essential,skill/competence,http://data.europa.eu/esco/skill/41a3240f-58ba...,conduct quality control analysis,05ebeb56-6ceb-488d-ba91-ed15088efc8e,41a3240f-58ba-44e2-a688-edeb21ea5c16,001,True,True
3,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""",...,bridge construction supervisor,essential,knowledge,http://data.europa.eu/esco/skill/44297bbb-306a...,mechanical tools,05ebeb56-6ceb-488d-ba91-ed15088efc8e,44297bbb-306a-46fa-ab1b-fa3fb327ebda,001,True,False
4,05ebeb56-6ceb-488d-ba91-ed15088efc8e,bridge construction supervisor,Bridge construction supervisors monitor the co...,421,42.1 Construction of roads and railways,42,42 Civil engineering,F,F CONSTRUCTION,"""construction supervisor""",...,bridge construction supervisor,essential,skill/competence,http://data.europa.eu/esco/skill/44e2c4c4-9fd2...,manage health and safety standards,05ebeb56-6ceb-488d-ba91-ed15088efc8e,44e2c4c4-9fd2-42d4-a133-d81360ded4bc,001,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,16b974f4-cb8c-436c-8c05-a16957c40131,fossil-fuel power plant operator,Fossil-fuel power plant operators operate and ...,351,"35.1 Electric power generation, transmission a...",35,"35 Electricity, gas, steam and air conditionin...",D,"D ELECTRICITY, GAS, STEAM AND AIR CONDITIONING...","""diesel generator operator""",...,fossil-fuel power plant operator,essential,knowledge,http://data.europa.eu/esco/skill/826c1cf4-582a...,fossil-fuel power plant operations,16b974f4-cb8c-436c-8c05-a16957c40131,826c1cf4-582a-4f1a-805f-98cc2b8000ba,005,True,False
96,16b974f4-cb8c-436c-8c05-a16957c40131,fossil-fuel power plant operator,Fossil-fuel power plant operators operate and ...,351,"35.1 Electric power generation, transmission a...",35,"35 Electricity, gas, steam and air conditionin...",D,"D ELECTRICITY, GAS, STEAM AND AIR CONDITIONING...","""diesel generator operator""",...,fossil-fuel power plant operator,essential,knowledge,http://data.europa.eu/esco/skill/c8ac1986-38fa...,electricity,16b974f4-cb8c-436c-8c05-a16957c40131,c8ac1986-38fa-43b2-89dc-50a4beeb420e,005,True,False
97,16b974f4-cb8c-436c-8c05-a16957c40131,fossil-fuel power plant operator,Fossil-fuel power plant operators operate and ...,351,"35.1 Electric power generation, transmission a...",35,"35 Electricity, gas, steam and air conditionin...",D,"D 

### 4.03 Save Results to CSV

In [25]:
# Set up output directory
output_dir = project_root / "data" / "silver" / "esco_nace_w_skills_csv"
output_dir.mkdir(parents=True, exist_ok=True)

# Create output filename
output_file = output_dir / f"{PROJECT_ID}_esco_nace_with_skills.csv"

# Save to CSV
df_with_evaluation.to_csv(output_file, index=False)

print(f"✓ Saved results to: {output_file}")
print(f"  Rows: {len(df_with_evaluation):,}")
print(f"  Columns: {len(df_with_evaluation.columns)}")
print(f"  Unique occupations: {df_with_evaluation['esco_id'].nunique()}")
print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")

✓ Saved results to: /Users/lauren/repos/PAD2Skills/data/silver/esco_nace_w_skills_csv/P511453_esco_nace_with_skills.csv
  Rows: 963
  Columns: 24
  Unique occupations: 44
  File size: 1780.5 KB
